In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!wget https://raw.githubusercontent.com/alexeygrigorev/datasets/master/car_fuel_efficiency.csv

--2025-11-11 08:57:26--  https://raw.githubusercontent.com/alexeygrigorev/datasets/master/car_fuel_efficiency.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 874188 (854K) [text/plain]
Saving to: ‘car_fuel_efficiency.csv’

car_fuel_efficiency 100%[===================>] 853.70K  --.-KB/s    in 0.05s   

2025-11-11 08:57:26 (18.4 MB/s) - ‘car_fuel_efficiency.csv’ saved [874188/874188]



In [3]:
df = pd.read_csv("car_fuel_efficiency.csv")

In [4]:
df.columns

Index(['engine_displacement', 'num_cylinders', 'horsepower', 'vehicle_weight',
       'acceleration', 'model_year', 'origin', 'fuel_type', 'drivetrain',
       'num_doors', 'fuel_efficiency_mpg'],
      dtype='object')

In [5]:
df = df.fillna(0)

In [6]:
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

In [7]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [8]:
y_train = df_train["fuel_efficiency_mpg"].values
y_val = df_val["fuel_efficiency_mpg"].values
y_test = df_test["fuel_efficiency_mpg"].values

In [9]:
del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']

In [10]:
df_train

,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,model_year,origin,fuel_type,drivetrain,num_doors
0,120,5.0,169.0,2966.679505,13.9,2005,USA,Gasoline,Front-wheel drive,-1.0
1,200,3.0,143.0,2950.822121,17.1,2013,Asia,Diesel,Front-wheel drive,-1.0
2,180,6.0,180.0,3078.221669,17.4,2007,USA,Gasoline,All-wheel drive,0.0
3,280,5.0,174.0,2797.991793,0.0,2016,USA,Diesel,All-wheel drive,0.0
4,250,4.0,133.0,2362.426930,16.3,2010,USA,Diesel,Front-wheel drive,-1.0
...,...,...,...,...,...,...,...,...,...,...
5817,230,3.0,176.0,3430.993044,17.9,2022,Europe,Diesel,All-wheel drive,0.0
5818,250,4.0,180.0,3067.664350,15.7,2010,Asia,Diesel,All-wheel drive,-1.0
5819,230,2.0,182.0,3041.964593,16.7,2010,Europe,Diesel,All-wheel drive,0.0
5820,180,7.0,147.0,2453.341430,15.2,2015,Europe,Gasoline,All-wheel drive,0.0


In [11]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import roc_auc_score
from sklearn.tree import export_text
from sklearn.tree import DecisionTreeRegressor

In [12]:
train_dicts = df_train.to_dict(orient='records')

In [13]:
dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)

In [14]:
X_train

array([[1.39000000e+01, 0.00000000e+00, 1.00000000e+00, ...,
        0.00000000e+00, 1.00000000e+00, 2.96667950e+03],
       [1.71000000e+01, 0.00000000e+00, 1.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 2.95082212e+03],
       [1.74000000e+01, 1.00000000e+00, 0.00000000e+00, ...,
        0.00000000e+00, 1.00000000e+00, 3.07822167e+03],
       ...,
       [1.67000000e+01, 1.00000000e+00, 0.00000000e+00, ...,
        1.00000000e+00, 0.00000000e+00, 3.04196459e+03],
       [1.52000000e+01, 1.00000000e+00, 0.00000000e+00, ...,
        1.00000000e+00, 0.00000000e+00, 2.45334143e+03],
       [1.41000000e+01, 0.00000000e+00, 1.00000000e+00, ...,
        0.00000000e+00, 1.00000000e+00, 2.83389943e+03]])

In [15]:
dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)

DecisionTreeRegressor(max_depth=1)

In [16]:
val_dicts = df_val.to_dict(orient='records')
X_val = dv.transform(val_dicts)

In [17]:
y_pred = dt.predict(X_val)
# roc_auc_score(y_val, y_pred)

In [18]:
print(export_text(dt, feature_names=list(dv.get_feature_names_out())))


|--- vehicle_weight <= 3022.11
|   |--- value: [16.88]
|--- vehicle_weight >  3022.11
|   |--- value: [12.94]



In [19]:
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor

In [20]:
rf = RandomForestRegressor(n_estimators=10, random_state=1)
rf.fit(X_train, y_train)

RandomForestRegressor(n_estimators=10, random_state=1)

In [21]:
y_pred_rf = rf.predict(X_val)
np.sqrt(mean_squared_error(y_val, y_pred_rf))

0.4599777557336148

In [22]:
n_est_acc = []
for n_est in np.linspace(10,200,20):
    rf = RandomForestRegressor(n_estimators=int(n_est), random_state=1, n_jobs=-1)
    rf.fit(X_train, y_train)
    y_pred_rf = rf.predict(X_val)
    print("n_est = ",n_est,"rmse = ", f"{np.sqrt(mean_squared_error(y_val, y_pred_rf)):.3f}")

n_est =  10.0 rmse =  0.460
n_est =  20.0 rmse =  0.454
n_est =  30.0 rmse =  0.451
n_est =  40.0 rmse =  0.448
n_est =  50.0 rmse =  0.446
n_est =  60.0 rmse =  0.445
n_est =  70.0 rmse =  0.445
n_est =  80.0 rmse =  0.445
n_est =  90.0 rmse =  0.445
n_est =  100.0 rmse =  0.444
n_est =  110.0 rmse =  0.443
n_est =  120.0 rmse =  0.444
n_est =  130.0 rmse =  0.443
n_est =  140.0 rmse =  0.443
n_est =  150.0 rmse =  0.443
n_est =  160.0 rmse =  0.443
n_est =  170.0 rmse =  0.443
n_est =  180.0 rmse =  0.442
n_est =  190.0 rmse =  0.443
n_est =  200.0 rmse =  0.443


In [23]:
for max_depth_i in np.linspace(10,25,4):
    rf_rmse = []
    for n_est in np.linspace(10,200,20):
        rf = RandomForestRegressor(n_estimators=int(n_est), max_depth=int(max_depth_i), random_state=1, n_jobs=-1)
        rf.fit(X_train, y_train)
        y_pred_rf = rf.predict(X_val)
        rf_rmse.append(np.sqrt(mean_squared_error(y_val, y_pred_rf)))

    print("max_depth = ", max_depth_i, "rmse = ", np.mean(rf_rmse) )

max_depth =  10.0 rmse =  0.44232130237115186
max_depth =  15.0 rmse =  0.44505999920137435
max_depth =  20.0 rmse =  0.4456441321803526
max_depth =  25.0 rmse =  0.44566060000292457


In [24]:
dt = RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1, n_jobs=-1)
dt.fit(X_train, y_train)
print()
print(dt.feature_importances_)


[1.14707165e-02 3.81809750e-04 3.11842085e-04 3.26932342e-03
 3.43693411e-04 3.36671988e-04 1.60402148e-02 3.18229841e-03
 2.35867094e-03 1.59113306e-03 4.76103046e-04 5.20358083e-04
 5.55151959e-04 9.59162013e-01]


In [27]:
importances = dt.feature_importances_
features = list(dv.get_feature_names_out())
features

['acceleration',
 'drivetrain=All-wheel drive',
 'drivetrain=Front-wheel drive',
 'engine_displacement',
 'fuel_type=Diesel',
 'fuel_type=Gasoline',
 'horsepower',
 'model_year',
 'num_cylinders',
 'num_doors',
 'origin=Asia',
 'origin=Europe',
 'origin=USA',
 'vehicle_weight']

In [28]:
for i, importance in enumerate(importances):
    print(f"Feature '{features[i]}': {importance:.4f}")

Feature 'acceleration': 0.0115
Feature 'drivetrain=All-wheel drive': 0.0004
Feature 'drivetrain=Front-wheel drive': 0.0003
Feature 'engine_displacement': 0.0033
Feature 'fuel_type=Diesel': 0.0003
Feature 'fuel_type=Gasoline': 0.0003
Feature 'horsepower': 0.0160
Feature 'model_year': 0.0032
Feature 'num_cylinders': 0.0024
Feature 'num_doors': 0.0016
Feature 'origin=Asia': 0.0005
Feature 'origin=Europe': 0.0005
Feature 'origin=USA': 0.0006
Feature 'vehicle_weight': 0.9592


In [29]:
import xgboost as xgb

In [30]:
features = list(dv.get_feature_names_out())
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=features)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=features)

In [34]:
xgb_params = {
    'eta': 0.1, 
    'max_depth': 6,
    'min_child_weight': 1,
    
    'objective': 'reg:squarederror',
    'nthread': 8,
    
    'seed': 1,
    'verbosity': 1,
}

model = xgb.train(xgb_params, dtrain, num_boost_round=10)

In [35]:
y_pred = model.predict(dval)
np.sqrt(mean_squared_error(y_val, y_pred))

1.0200885118810736